In [3]:
import gdsfactory as gf
import numpy as np
import cspdk.si220.cband
from typing import List
from axiomatic.pic_helpers import plot_circuit
cspdk.si220.cband.activate_pdk()

pdk = gf.get_active_pdk()
c = gf.Component()

def add_drop(coupling_lengths: List[float], target_fsr: float, coupling_gap: float, center_wvl: float) -> gf.Component:

    bend_radius = 5  # Bend radius in micrometers
    n_eff = 2.38
    n_g = 4.3

    cavity_length_target_fsr = center_wvl**2 / (n_g * target_fsr * 1e-3)

    m = round(n_eff * cavity_length_target_fsr / center_wvl)
    ring_length_target = m * center_wvl / n_eff  # Actual round-trip length for resonance at 1.55 um
    length_arc = 2 * np.pi * bend_radius

    c = gf.Component()
    coupler = lambda L: pdk.get_component("coupler", length=L, gap=coupling_gap)

    # Calculate vertical straight section of the ring from dc lengths
    dcs = []
    dc_lengths = []
    for length in coupling_lengths:
        dc = c << coupler(length)
        dc_length = 2 * dc.cell.info['length'] + length
        dcs.append(dc)
        dc_lengths.append(dc_length)
    y_dist = (cavity_length_target_fsr - (4*np.pi/2*bend_radius + sum(dc_lengths) + abs(coupling_lengths[-1] - coupling_lengths[0])))/2 

    # Place couplers
    dy = abs(dcs[-1].ports["o2"].center[1] - dcs[-1].ports["o1"].center[1])
    for i, dc in enumerate(dcs):
        # Get bounding box center x-coordinate
        bbox = dc.bbox()
        center_x = (bbox.left + bbox.right) / 2

        # Calculate the move offsets
        move_x = -center_x

        # Calculate the move offsets
        move_y = i * (y_dist + dy + 2*bend_radius)

        # Apply the move to center horizontally and stack vertically
        dc.move((move_x, move_y))

    upper_right_conn = dcs[0].ports["o3"]
    lower_right_conn = dcs[-1].ports["o4"]
    upper_left_conn = dcs[-1].ports["o1"]
    lower_left_conn = dcs[0].ports["o2"]
    
    right_ring_conn = gf.routing.route_single(
            component=c,
            port1=upper_right_conn,
            port2=lower_right_conn,
            bend="bend_euler",
            cross_section="strip"
        )

    left_ring_conn = gf.routing.route_single(
        component=c,
        port1=upper_left_conn,
        port2=lower_left_conn,
        bend="bend_euler",
        cross_section="strip"
    )

    return c

def multi_add_drop(num_rings: int, x_spacing: float, coupling_lengths: List[List[float]], target_fsrs: List[float], coupling_gaps: List[float], center_wvls: List[float]) -> gf.Component:
    c = gf.Component()
    
    # y offset to align all rings along the same bus waveguide
    y_offset = 0.0

    # Store each add-drop ring
    rings = []

    for i in range(num_rings):
        add_drop_ring = add_drop(
            coupling_lengths=coupling_lengths[i],
            target_fsr=target_fsrs[i],
            coupling_gap=coupling_gaps[i],
            center_wvl=center_wvls[i]
        )
        add_drop_ref = c << add_drop_ring

        # Position along x
        x_offset = i * x_spacing

        # Move to align on the same bus line
        add_drop_ref.move((x_offset, y_offset))
        rings.append(add_drop_ref)

    # Optionally, add input/output ports at the bus level
    # For example, add port at leftmost and rightmost add-drop ring
    c.add_port("bus_in", port=rings[0].ports["o1"])
    c.add_port("bus_out", port=rings[-1].ports["o2"])

    return c


if __name__ == "__main__":
    c = multi_add_drop(
        num_rings=3,
        x_spacing=200,
        coupling_lengths=[[20, 30], [25, 35], [22, 28]],
        target_fsrs=[1, 1, 1],
        coupling_gaps=[0.2, 0.2, 0.2],
        center_wvls=[1.55, 1.55, 1.55]
    )
    plot_circuit(c)


KeyError: "key='o1' is not a valid port name or index. Make sure the instance is an array when giving it a tuple. Available ports: []"